# DarkIR — Physics-Guided Loss Functions (Fine-tuning)

This notebook fine-tunes the pretrained **DarkIR** model with three new physics-guided loss terms:

| Loss | Physical prior | Key operation |
|---|---|---|
| `RetinexLoss` | Retinex: `I = R ⊙ L` | Max-pool illumination estimation + TV smoothness |
| `BlurAwareGradientLoss` | Blur formation: `y = x ⊗ k` | Re-blur gradient consistency with learnable Gaussian PSF |
| `PhaseEnhancedFrequencyLoss` | Fourier phase carries structure | Phase-gradient supervision on top of amplitude loss |

No architectural changes are made to DarkIR — these are **training-only** additions with zero inference overhead.

**Progressive integration schedule**

| Phase | Epochs | Active losses |
|---|---|---|
| 1 — Baseline | 1 – `PHASE2_START-1` | L1 + EdgeLoss |
| 2 — + Retinex | `PHASE2_START` – `PHASE3_START-1` | + `RetinexLoss` |
| 3 — + Blur | `PHASE3_START` – `PHASE4_START-1` | + `BlurAwareGradientLoss` |
| 4 — Full | `PHASE4_START` – end | + `PhaseEnhancedFrequencyLoss` |

**Based on:** `DarkIR_Kaggle.ipynb` (inference notebook).  
**Accelerator:** GPU T4 x2 or P100 recommended.

## 1. Install dependencies

In [ ]:
!pip install -q ptflops lpips pytorch-msssim pyiqa einops

## 2. Clone repository and set paths

In [ ]:
import os

REPO_DIR = '/kaggle/working/DarkIR'

if not os.path.isdir(REPO_DIR):
    !git clone https://github.com/Fundacion-Cidaut/DarkIR.git {REPO_DIR}
else:
    print('Repo already cloned.')

import sys
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

## 3. Download pretrained weights

Fine-tuning starts from the pretrained checkpoint — this gives us a strong initialisation and means we only need 50–100 epochs to see the effect of the new losses.

In [ ]:
from huggingface_hub import hf_hub_download

# 'DarkIR_32width.pt'  →  DarkIR-m  (3.3 M params, lighter)
# 'DarkIR_64width.pt'  →  DarkIR-l  (13 M params, stronger baseline)
MODEL_VARIANT = 'DarkIR_64width.pt'

weights_path = hf_hub_download(repo_id='Cidaut/DarkIR', filename=MODEL_VARIANT)
print('Pretrained weights at:', weights_path)

## 4. Imports and model configuration

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.transforms import Resize
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm
from pathlib import Path
import glob

device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')
print('Using device:', device)

# Must match the downloaded checkpoint variant
MODEL_WIDTH = 64  # set to 32 if using DarkIR_32width.pt

model_cfg = dict(
    name             = 'DarkIR',
    img_channels     = 3,
    width            = MODEL_WIDTH,
    middle_blk_num_enc = 2,
    middle_blk_num_dec = 2,
    enc_blk_nums     = [1, 2, 3],
    dec_blk_nums     = [3, 1, 1],
    dilations        = [1, 4, 9],
    extra_depth_wise = True,
)

## 5. Build model

We load the pretrained weights without DDP (single device, no `module.` prefix needed).

In [ ]:
from archs.DarkIR import DarkIR

def build_model(cfg, weights_path, device):
    model = DarkIR(
        img_channel        = cfg['img_channels'],
        width              = cfg['width'],
        middle_blk_num_enc = cfg['middle_blk_num_enc'],
        middle_blk_num_dec = cfg['middle_blk_num_dec'],
        enc_blk_nums       = cfg['enc_blk_nums'],
        dec_blk_nums       = cfg['dec_blk_nums'],
        dilations          = cfg['dilations'],
        extra_depth_wise   = cfg['extra_depth_wise'],
    )
    checkpoint = torch.load(weights_path, map_location='cpu', weights_only=False)
    # 'params' stores weights without the DDP 'module.' prefix
    model.load_state_dict(checkpoint['params'])
    print('Pretrained weights loaded.')
    model.to(device)
    return model

model = build_model(model_cfg, weights_path, device)

---
## 6. Physics-Guided Loss Functions

All three losses are defined inline here so the notebook is fully self-contained.
They are **training-only** — they add zero overhead at inference time.

### 6.1 Retinex Decomposition Loss

Retinex theory: the observed image `I` decomposes as `I = R ⊙ L`, where `L` is a slowly-varying illumination map and `R` is the reflectance (scene content).

Rather than training a dedicated decomposition network, we estimate `L` heuristically via a large spatial max-pool over the predicted output, then enforce:

1. **Illumination smoothness** — `||∇L||₁` (total variation)
2. **Reconstruction consistency** — `||R ⊙ L − I_pred||₁`

This acts as an architecture-agnostic regulariser.

In [ ]:
class RetinexLoss(nn.Module):
    """
    Physics-guided Retinex decomposition loss.

    Illumination L is estimated by taking the per-pixel maximum across colour
    channels and then smoothing with a large max-pool.  Reflectance R = I / L.

    Two terms are enforced:
      smooth_loss : total-variation regularisation on L  (slow spatial variation)
      recon_loss  : ||R * L - I_pred||_1                 (Retinex identity)
    """

    def __init__(self, weight=0.1, pool_kernel=15,
                 smooth_weight=0.5, recon_weight=0.5):
        super().__init__()
        self.weight        = weight
        self.smooth_weight = smooth_weight
        self.recon_weight  = recon_weight
        pad = pool_kernel // 2
        # Max-pool as a local-maximum illumination estimator
        self.pool = nn.MaxPool2d(pool_kernel, stride=1, padding=pad)

    def forward(self, x_pred):
        # ── illumination estimate ────────────────────────────────────────────
        # Step 1: max across colour channels → (B, 1, H, W)
        L = x_pred.max(dim=1, keepdim=True).values
        # Step 2: smooth spatially → (B, 1, H, W)
        L = self.pool(L)
        # Clamp to avoid near-zero division
        L = L.clamp(min=1e-6)
        # Expand back to 3 channels for broadcasting
        L3 = L.expand_as(x_pred)

        R  = x_pred / L3                                  # reflectance

        # ── term 1: illumination total-variation smoothness ──────────────────
        smooth_h = torch.abs(L[:, :, 1:, :] - L[:, :, :-1, :]).mean()
        smooth_w = torch.abs(L[:, :, :, 1:] - L[:, :, :, :-1]).mean()
        smooth_loss = smooth_h + smooth_w

        # ── term 2: reconstruction consistency ──────────────────────────────
        recon_loss = F.l1_loss(R * L3, x_pred)

        return self.weight * (
            self.smooth_weight * smooth_loss +
            self.recon_weight  * recon_loss
        )

# quick sanity check
_t = torch.rand(2, 3, 64, 64)
print('RetinexLoss:', RetinexLoss()(_t).item())

### 6.2 Blur-Aware Gradient Loss (Phase 1 — Gaussian PSF)

Blur formation model: `y = x ⊗ k`, where `y` is the blurred observation and `k` is the point-spread function (PSF).

We enforce **re-blur gradient consistency**:

$$\|\nabla(\hat{x} \otimes k_{\text{est}}) - \nabla y\|_1$$

meaning the gradients of the re-blurred prediction should match the gradients of the original input. `k_est` is a parametric isotropic Gaussian with a **learnable sigma**, updated by backprop alongside the model.

> **Phase 1 (implemented here)** — Gaussian PSF with learnable `σ`. Safe and stable.  
> **Phase 2** (extension) — FFT-based kernel estimation. More expressive but numerically sensitive; requires thresholding to prevent division-by-near-zero amplification.

In [ ]:
class BlurAwareGradientLoss(nn.Module):
    """
    Blur-aware gradient loss with a learnable Gaussian PSF (Phase 1).

    Enforces:  ||∇(x_pred ⊗ k_est) − ∇y_input||_1

    k_est is a 2-D isotropic Gaussian kernel whose sigma is a trainable
    parameter, updated via backprop.  Gradients are computed with fixed
    Sobel filters applied per colour channel.
    """

    def __init__(self, weight=0.05, init_sigma=1.5, kernel_size=11):
        super().__init__()
        self.weight      = weight
        self.kernel_size = kernel_size
        # log-parameterised sigma: always positive, updated by backprop
        self.log_sigma = nn.Parameter(torch.tensor(float(init_sigma)).log())

        # Fixed Sobel kernels registered as buffers (not trained)
        sx = torch.tensor([[-1., 0., 1.],
                            [-2., 0., 2.],
                            [-1., 0., 1.]])
        sy = torch.tensor([[-1., -2., -1.],
                            [ 0.,  0.,  0.],
                            [ 1.,  2.,  1.]])
        self.register_buffer('sobel_x', sx.view(1, 1, 3, 3))
        self.register_buffer('sobel_y', sy.view(1, 1, 3, 3))

    # ── helpers ──────────────────────────────────────────────────────────────

    def _gaussian_kernel(self, device):
        """Build a normalised 2-D Gaussian kernel from the current sigma."""
        sigma = self.log_sigma.exp().clamp(0.3, 5.0)
        k     = self.kernel_size
        coords = torch.arange(k, dtype=torch.float32, device=device) - k // 2
        g1d    = torch.exp(-0.5 * (coords / sigma) ** 2)
        g2d    = g1d.unsqueeze(0) * g1d.unsqueeze(1)   # outer product
        return (g2d / g2d.sum()).view(1, 1, k, k)

    def _sobel(self, x):
        """Gradient magnitude via Sobel, applied independently per channel."""
        B, C, H, W = x.shape
        xf = x.view(B * C, 1, H, W)
        gx = F.conv2d(xf, self.sobel_x, padding=1)
        gy = F.conv2d(xf, self.sobel_y, padding=1)
        return (gx ** 2 + gy ** 2 + 1e-8).sqrt().view(B, C, H, W)

    def _blur(self, x):
        """Apply the Gaussian PSF to x via depthwise conv."""
        B, C, H, W = x.shape
        k = self._gaussian_kernel(x.device).expand(C, 1, self.kernel_size, self.kernel_size)
        return F.conv2d(x, k, padding=self.kernel_size // 2, groups=C)

    # ── forward ──────────────────────────────────────────────────────────────

    def forward(self, x_pred, y_input):
        """
        Args:
            x_pred  : enhanced output from DarkIR        (B, 3, H, W)
            y_input : corresponding low-light input image (B, 3, H, W)
        """
        grad_reblurred = self._sobel(self._blur(x_pred))
        grad_input     = self._sobel(y_input)
        return self.weight * F.l1_loss(grad_reblurred, grad_input)

# quick sanity check
_p = torch.rand(2, 3, 64, 64)
_y = torch.rand(2, 3, 64, 64)
print('BlurAwareGradientLoss:', BlurAwareGradientLoss()(_p, _y).item())

### 6.3 Phase-Enhanced Frequency Loss

DarkIR's `FreMLP` already operates on Fourier **amplitude**, reflecting the observation that illumination degradation primarily affects amplitude while **phase** (structural information) is relatively preserved.

This loss extends that with explicit phase supervision:

$$\mathcal{L}_{\text{freq}} = \lambda_{\text{amp}} \cdot \|A_{\hat{x}} - A_{y}\|_1 + \lambda_{\text{phase}} \cdot \|\nabla_{f}\phi_{\hat{x}} - \nabla_{f}\phi_{y}\|_1$$

where `∇_f` denotes finite differences along the frequency axes (a frequency-domain gradient), which avoids 2π wrapping issues inherent in direct phase comparison. A high-pass mask zeros out DC and near-DC components where phase is noisy and uninformative.

In [ ]:
class PhaseEnhancedFrequencyLoss(nn.Module):
    """
    Phase-enhanced frequency loss.

    Extends amplitude-only frequency supervision (as in DarkIR's FrequencyLoss)
    with explicit phase-gradient supervision in the Fourier domain.

    A high-pass mask suppresses DC and near-DC frequencies where phase is
    unreliable.  Phase is represented as unit complex vectors e^{iφ} = F/|F|,
    and phase gradients are taken along both frequency axes.
    """

    def __init__(self, weight=0.05, amp_weight=0.5, phase_weight=0.5,
                 high_pass_ratio=0.1):
        """
        Args:
            weight          : overall loss scale
            amp_weight      : relative weight for the amplitude term
            phase_weight    : relative weight for the phase-gradient term
            high_pass_ratio : fraction of the frequency grid masked out at
                              low frequencies (applied to both H and W axes)
        """
        super().__init__()
        self.weight          = weight
        self.amp_weight      = amp_weight
        self.phase_weight    = phase_weight
        self.high_pass_ratio = high_pass_ratio

    # ── helpers ──────────────────────────────────────────────────────────────

    @staticmethod
    def _phase_unit(fft_tensor):
        """Unit complex vector e^{iφ} = F / |F|. Avoids 2π wrapping."""
        amp = fft_tensor.abs().clamp(min=1e-8)
        return fft_tensor / amp

    @staticmethod
    def _freq_grad(p):
        """Finite differences along frequency axes (H-freq and W-freq)."""
        dh = p[:, :, 1:, :] - p[:, :, :-1, :]   # along H-frequency axis
        dw = p[:, :, :, 1:] - p[:, :, :, :-1]   # along W-frequency axis
        return dh, dw

    def _high_pass_mask(self, shape, device):
        """Ones everywhere except the low-frequency corner."""
        _, _, H, W_r = shape
        mask = torch.ones(1, 1, H, W_r, device=device)
        r_h  = max(1, int(H   * self.high_pass_ratio))
        r_w  = max(1, int(W_r * self.high_pass_ratio))
        mask[:, :, :r_h, :r_w] = 0.0
        return mask

    # ── forward ──────────────────────────────────────────────────────────────

    def forward(self, pred, target):
        """
        Args:
            pred   : enhanced output from DarkIR  (B, 3, H, W)  in [0, 1]
            target : ground-truth image            (B, 3, H, W)  in [0, 1]
        """
        pred_fft   = torch.fft.rfft2(pred,   norm='backward')
        target_fft = torch.fft.rfft2(target, norm='backward')

        mask = self._high_pass_mask(pred_fft.shape, pred_fft.device)

        # ── amplitude term ───────────────────────────────────────────────────
        amp_loss = F.l1_loss(pred_fft.abs() * mask,
                             target_fft.abs() * mask)

        # ── phase-gradient term ──────────────────────────────────────────────
        pred_ph   = self._phase_unit(pred_fft)
        target_ph = self._phase_unit(target_fft)

        pred_dh,   pred_dw   = self._freq_grad(pred_ph)
        target_dh, target_dw = self._freq_grad(target_ph)

        # Use real part of the gradient difference (imaginary part gives
        # equivalent information and using both doubles cost for no gain)
        phase_loss = (
            F.l1_loss(pred_dh.real * mask[:, :, 1:, :],
                      target_dh.real * mask[:, :, 1:, :]) +
            F.l1_loss(pred_dw.real * mask[:, :, :, 1:],
                      target_dw.real * mask[:, :, :, 1:])
        )

        return self.weight * (
            self.amp_weight   * amp_loss +
            self.phase_weight * phase_loss
        )

# quick sanity check
_p = torch.rand(2, 3, 64, 64)
_t = torch.rand(2, 3, 64, 64)
print('PhaseEnhancedFrequencyLoss:', PhaseEnhancedFrequencyLoss()(_p, _t).item())

---
## 7. Dataset setup

Fine-tuning requires paired low-light / normal-light images.

**Recommended datasets** (all publicly available):  
- LOL-Blur test split (1 800 pairs) — best match for DarkIR's training distribution  
- LOLv2-real / LOLv2-synth (100 test pairs each)  
- LSRW-Nikon or LSRW-Huawei

**To upload on Kaggle:** *Add Data → Upload dataset* and point `LOW_TRAIN_DIR` / `HIGH_TRAIN_DIR` at the resulting `/kaggle/input/<dataset-name>/` paths.

The cell below falls back to the sample images bundled in the repo so you can verify the pipeline runs before adding a full dataset.

In [ ]:
# ── Set these to your uploaded dataset paths ─────────────────────────────────
LOW_TRAIN_DIR  = f'{REPO_DIR}/assets/qualis/inputs'   # ← replace with real data
HIGH_TRAIN_DIR = f'{REPO_DIR}/assets/qualis/results'  # ← replace with real data
# ─────────────────────────────────────────────────────────────────────────────

SUPPORTED = {'.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG'}

low_paths  = sorted([p for p in Path(LOW_TRAIN_DIR).iterdir()  if p.suffix in SUPPORTED])
high_paths = sorted([p for p in Path(HIGH_TRAIN_DIR).iterdir() if p.suffix in SUPPORTED])

assert len(low_paths) == len(high_paths) and len(low_paths) > 0, \
    f'Found {len(low_paths)} low and {len(high_paths)} high images — check your paths.'

print(f'Training pairs: {len(low_paths)}')

In [ ]:
# Import the repo's paired-image dataset class directly
from data.dataset_reader.datapipeline import MyDataset_Crop

CROP_SIZE  = 256   # spatial crop size during training
BATCH_SIZE = 4     # reduce to 2 if OOM
NUM_WORKERS = 2

to_tensor = transforms.ToTensor()

train_dataset = MyDataset_Crop(
    images_low  = [str(p) for p in low_paths],
    images_high = [str(p) for p in high_paths],
    cropsize     = CROP_SIZE,
    tensor_transform = to_tensor,
    test         = False,
)

train_loader = DataLoader(
    train_dataset,
    batch_size  = BATCH_SIZE,
    shuffle     = True,
    num_workers = NUM_WORKERS,
    pin_memory  = device.type == 'cuda',
    drop_last   = True,
)

print(f'Batches per epoch: {len(train_loader)}')

---
## 8. Training configuration

Loss weights and phase boundaries follow the recommendations in the proposal:
- Start with a low weight (`0.1`) for Retinex and increase progressively.
- Keep phase loss weight low (`beta = 0.1` initial) to avoid destabilising training.
- The blur-aware gradient loss weight (`0.05`) balances its stronger gradient signal.

In [ ]:
# ── Total epochs and progressive phase boundaries ────────────────────────────
TOTAL_EPOCHS  = 80
PHASE2_START  = 21   # epoch at which RetinexLoss is added
PHASE3_START  = 41   # epoch at which BlurAwareGradientLoss is added
PHASE4_START  = 61   # epoch at which PhaseEnhancedFrequencyLoss is added

# ── Loss weights ─────────────────────────────────────────────────────────────
W_L1      = 1.0     # reconstruction L1 (always active)
W_EDGE    = 0.05    # edge sharpness    (always active)
W_RETINEX = 0.10    # Retinex loss
W_BLUR    = 0.05    # blur-aware gradient loss
W_PHASE   = 0.05    # phase-enhanced frequency loss

# ── Instantiate losses ───────────────────────────────────────────────────────
# Baseline losses (imported from the repo)
from losses.loss import EdgeLoss

loss_l1   = nn.L1Loss().to(device)
loss_edge = EdgeLoss(rank=device, loss_weight=W_EDGE, criterion='l2').to(device)

# Physics-guided losses (defined above)
loss_retinex = RetinexLoss(weight=W_RETINEX).to(device)
loss_blur    = BlurAwareGradientLoss(weight=W_BLUR).to(device)
loss_phase   = PhaseEnhancedFrequencyLoss(weight=W_PHASE).to(device)

# ── Collect all parameters: model + learnable sigma in blur loss ─────────────
all_params = list(model.parameters()) + list(loss_blur.parameters())

optimizer = optim.AdamW(all_params, lr=2e-5, weight_decay=1e-4, betas=(0.9, 0.999))
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-7)

# Directory for saving checkpoints
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(CKPT_DIR, exist_ok=True)

print('Losses and optimiser ready.')
print(f'Learnable blur sigma initialised to: {loss_blur.log_sigma.exp().item():.3f}')

---
## 9. Fine-tuning loop

Each phase prints which losses are active so you can verify the progressive schedule at a glance.  
Checkpoints are saved at the end of every 10 epochs and are compatible with the original inference code (stored under the `'params'` key).

In [ ]:
def get_active_phase(epoch):
    """Return the current integration phase (1–4) for a given epoch."""
    if epoch >= PHASE4_START: return 4
    if epoch >= PHASE3_START: return 3
    if epoch >= PHASE2_START: return 2
    return 1


PHASE_LABELS = {
    1: 'L1 + Edge',
    2: 'L1 + Edge + Retinex',
    3: 'L1 + Edge + Retinex + BlurGrad',
    4: 'L1 + Edge + Retinex + BlurGrad + PhaseFreq  [FULL]',
}

history = []   # list of dicts for post-hoc plotting

for epoch in range(1, TOTAL_EPOCHS + 1):
    phase = get_active_phase(epoch)

    # Print a header whenever the phase changes
    if epoch in (1, PHASE2_START, PHASE3_START, PHASE4_START):
        print(f'\n── Phase {phase} (epoch {epoch}) ─────────────────────────────────')
        print(f'   Active losses: {PHASE_LABELS[phase]}')

    model.train()
    running = dict(total=0., l1=0., edge=0., retinex=0., blur=0., phase_freq=0.)

    pbar = tqdm(train_loader, desc=f'Epoch {epoch}/{TOTAL_EPOCHS}', leave=False)
    for high_batch, low_batch in pbar:
        high_batch = high_batch.to(device)
        low_batch  = low_batch.to(device)

        optimizer.zero_grad()

        enhanced = model(low_batch, side_loss=False)
        enhanced = torch.clamp(enhanced, 0., 1.)

        # ── baseline losses (always active) ──────────────────────────────────
        l_l1   = W_L1 * loss_l1(enhanced, high_batch)
        l_edge = loss_edge(enhanced, high_batch)
        loss   = l_l1 + l_edge

        # ── Phase 2+: Retinex loss ────────────────────────────────────────────
        l_retinex = torch.tensor(0., device=device)
        if phase >= 2:
            l_retinex = loss_retinex(enhanced)
            loss = loss + l_retinex

        # ── Phase 3+: blur-aware gradient loss ────────────────────────────────
        l_blur = torch.tensor(0., device=device)
        if phase >= 3:
            l_blur = loss_blur(enhanced, low_batch)
            loss   = loss + l_blur

        # ── Phase 4: phase-enhanced frequency loss ────────────────────────────
        l_phase_freq = torch.tensor(0., device=device)
        if phase >= 4:
            l_phase_freq = loss_phase(enhanced, high_batch)
            loss         = loss + l_phase_freq

        loss.backward()
        # Gradient clipping prevents instability, especially during early
        # epochs of the blur loss where sigma is still converging
        torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
        optimizer.step()

        # Accumulate for epoch summary
        n = high_batch.size(0)
        running['total']      += loss.item()      * n
        running['l1']         += l_l1.item()      * n
        running['edge']       += l_edge.item()    * n
        running['retinex']    += l_retinex.item() * n
        running['blur']       += l_blur.item()    * n
        running['phase_freq'] += l_phase_freq.item() * n

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    scheduler.step()

    N  = len(train_dataset)
    ep = {k: v / N for k, v in running.items()}
    ep['epoch'] = epoch
    ep['lr']    = scheduler.get_last_lr()[0]
    ep['sigma'] = loss_blur.log_sigma.exp().item()
    history.append(ep)

    print(f'Epoch {epoch:3d} | total={ep["total"]:.4f}  '
          f'l1={ep["l1"]:.4f}  edge={ep["edge"]:.4f}  '
          f'retinex={ep["retinex"]:.4f}  blur={ep["blur"]:.4f}  '
          f'phase={ep["phase_freq"]:.4f}  σ={ep["sigma"]:.3f}  '
          f'lr={ep["lr"]:.2e}')

    # Save checkpoint every 10 epochs
    if epoch % 10 == 0:
        ckpt_path = os.path.join(CKPT_DIR, f'DarkIR_physics_ep{epoch:03d}.pt')
        torch.save({
            'epoch'     : epoch,
            'params'    : model.state_dict(),      # 'params' key is compatible with the original inference code
            'optimizer' : optimizer.state_dict(),
            'scheduler' : scheduler.state_dict(),
            'blur_sigma': loss_blur.log_sigma.item(),
        }, ckpt_path)
        print(f'   Checkpoint saved → {ckpt_path}')

print('\nFine-tuning complete.')

### Training history plots

In [ ]:
epochs     = [h['epoch']     for h in history]
loss_total = [h['total']     for h in history]
loss_ret   = [h['retinex']   for h in history]
loss_blur_ = [h['blur']      for h in history]
loss_ph    = [h['phase_freq']for h in history]
sigmas     = [h['sigma']     for h in history]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, loss_total, label='Total')
axes[0].set_title('Total loss')
axes[0].set_xlabel('Epoch')
for x, label in [(PHASE2_START, 'P2'), (PHASE3_START, 'P3'), (PHASE4_START, 'P4')]:
    axes[0].axvline(x, ls='--', c='gray', lw=0.8, label=label)
axes[0].legend(fontsize=8)

axes[1].plot(epochs, loss_ret,   label='Retinex')
axes[1].plot(epochs, loss_blur_, label='BlurGrad')
axes[1].plot(epochs, loss_ph,    label='PhaseFreq')
axes[1].set_title('Physics loss components')
axes[1].set_xlabel('Epoch')
axes[1].legend(fontsize=8)

axes[2].plot(epochs, sigmas)
axes[2].set_title('Learned blur sigma (σ)')
axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig('/kaggle/working/training_history.png', dpi=120)
plt.show()

---
## 10. Inference with the fine-tuned model

Same utilities as `DarkIR_Kaggle.ipynb` — no changes needed at inference time.

In [ ]:
pil_to_tensor = transforms.ToTensor()
tensor_to_pil = transforms.ToPILImage()


def load_image(path):
    return pil_to_tensor(Image.open(path).convert('RGB')).unsqueeze(0)


def pad_to_multiple(tensor, multiple=8):
    _, _, H, W = tensor.shape
    ph = (multiple - H % multiple) % multiple
    pw = (multiple - W % multiple) % multiple
    return F.pad(tensor, (0, pw, 0, ph), value=0)


@torch.no_grad()
def enhance_image(model, tensor, device, max_side=1500):
    tensor = tensor.to(device)
    _, _, H, W = tensor.shape
    if H >= max_side or W >= max_side:
        tensor  = Resize((H // 2, W // 2))(tensor)
        resized = True
    else:
        resized = False
    padded = pad_to_multiple(tensor)
    output = model(padded, side_loss=False)
    oh, ow = (H // 2, W // 2) if resized else (H, W)
    output = output[:, :, :oh, :ow]
    if resized:
        output = Resize((H, W))(output)
    return torch.clamp(output, 0., 1.).cpu()


def tensor_to_numpy(t):
    return (t.squeeze(0).permute(1, 2, 0).numpy() * 255).clip(0, 255).astype(np.uint8)


print('Inference utilities ready.')

In [ ]:
model.eval()

# ── Change to any low-light image ────────────────────────────────────────────
IMAGE_PATH = f'{REPO_DIR}/assets/teaser/0085_low.png'
# ─────────────────────────────────────────────────────────────────────────────

inp_tensor = load_image(IMAGE_PATH)
out_tensor = enhance_image(model, inp_tensor, device)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].imshow(tensor_to_numpy(inp_tensor)); axes[0].set_title('Input (low-light)'); axes[0].axis('off')
axes[1].imshow(tensor_to_numpy(out_tensor)); axes[1].set_title('Fine-tuned DarkIR'); axes[1].axis('off')
plt.tight_layout()
plt.show()

tensor_to_pil(out_tensor.squeeze(0)).save('/kaggle/working/finetuned_result.png')
print('Saved to /kaggle/working/finetuned_result.png')

---
## 11. (Optional) Quantitative evaluation

Computes PSNR / SSIM / LPIPS over a paired test set. Set `LOW_TEST_DIR` and `HIGH_TEST_DIR` to your test-split paths.

In [ ]:
# ── Set these to your test-split paths ───────────────────────────────────────
LOW_TEST_DIR  = '/kaggle/input/your-dataset/test/low'
HIGH_TEST_DIR = '/kaggle/input/your-dataset/test/high'
# ─────────────────────────────────────────────────────────────────────────────

if not (os.path.isdir(LOW_TEST_DIR) and os.path.isdir(HIGH_TEST_DIR)):
    print('Test dirs not set — skipping evaluation.')
else:
    from lpips import LPIPS
    from pytorch_msssim import ssim as calc_ssim_fn

    low_test  = sorted([p for p in Path(LOW_TEST_DIR).iterdir()  if p.suffix in SUPPORTED])
    high_test = sorted([p for p in Path(HIGH_TEST_DIR).iterdir() if p.suffix in SUPPORTED])
    assert len(low_test) == len(high_test)
    print(f'Evaluating on {len(low_test)} pairs...')

    lpips_fn = LPIPS(net='vgg', verbose=False).to(device)
    psnr_list, ssim_list, lpips_list = [], [], []

    model.eval()
    for lp, hp in tqdm(zip(low_test, high_test), total=len(low_test), desc='Evaluating'):
        low_t  = load_image(lp).to(device)
        high_t = load_image(hp).to(device)
        with torch.no_grad():
            enh = enhance_image(model, low_t, device).to(device)
            mse = torch.mean((high_t - enh) ** 2).item()
            psnr_list.append(20 * np.log10(1.0 / np.sqrt(mse + 1e-8)))
            ssim_list.append(calc_ssim_fn(enh, high_t, data_range=1.0, size_average=True).item())
            lpips_list.append(lpips_fn(enh * 2 - 1, high_t * 2 - 1).item())

    print(f'\nResults over {len(low_test)} pairs:')
    print(f'  PSNR  : {np.mean(psnr_list):.4f} dB')
    print(f'  SSIM  : {np.mean(ssim_list):.4f}')
    print(f'  LPIPS : {np.mean(lpips_list):.4f}')

---
## Notes

**Loading a saved checkpoint for inference:**
```python
ckpt = torch.load('/kaggle/working/checkpoints/DarkIR_physics_ep080.pt', map_location='cpu')
model.load_state_dict(ckpt['params'])   # compatible with original inference code
```

**Extending to Phase 2 blur loss (FFT-based kernel estimation):**  
Replace `_gaussian_kernel` in `BlurAwareGradientLoss` with an FFT-based approach:
```python
# Numerically unstable — threshold to prevent near-zero division
K_est = torch.fft.rfft2(y) / (torch.fft.rfft2(x_pred).abs().clamp(min=1e-3))
```
Monitor training loss carefully; clip gradients aggressively (`max_norm=0.5`).

**Hyperparameter sensitivity:**  
With 7 loss terms in total, consider *uncertainty-based automatic weighting* (Kendall et al., NeurIPS 2018) as a drop-in replacement for the fixed `W_*` constants above.

**Expected gains:**  
Loss-only modifications typically yield **+0.2 – 0.5 dB PSNR** over the pretrained baseline. A +0.8 dB target is optimistic; aim for +0.3 dB as the minimum viable result.